In [8]:
import streamlit as st
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import plotly.graph_objects as go
from streamlit_option_menu import option_menu

# ------------------------------------------------
# 1️⃣ Page Configuration & Project Theme
# ------------------------------------------------
st.set_page_config(page_title="🚫 Cyberbullying & Hate Speech Detection", layout="wide")

st.markdown("""
    <style>
    .stApp { background-color: #f8fafc; }
    .main-card {
        background-color: white;
        padding: 25px;
        border-radius: 15px;
        box-shadow: 0 4px 6px rgba(0,0,0,0.05);
        margin-bottom: 20px;
    }
    .legal-card {
        background-color: #ffffff;
        padding: 15px;
        border-radius: 8px;
        border-left: 5px solid #1e40af;
        margin-bottom: 15px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.05);
    }
    .section-header { color: #1e3a8a; font-weight: bold; margin-bottom: 15px; }
    </style>
    """, unsafe_allow_html=True)

# ------------------------------------------------
# 2️⃣ Backend: Model & Logic
# ------------------------------------------------
@st.cache_resource
def load_model():
    tokenizer = AutoTokenizer.from_pretrained("unitary/toxic-bert")
    model = AutoModelForSequenceClassification.from_pretrained("unitary/toxic-bert")
    return tokenizer, model

tokenizer, model = load_model()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

labels = ["Toxic", "Severe Toxic", "Obscene", "Threat", "Insult", "Identity Hate"]

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs).logits
        probs = torch.sigmoid(outputs).cpu().numpy()[0]
    return probs

# ------------------------------------------------
# 3️⃣ Sidebar Navigation
# ------------------------------------------------
with st.sidebar:
    st.markdown("<h1 style='text-align: center;'>🛡️🚫 Cyberbullying & Hate Speech Detection</h1>", unsafe_allow_html=True)
    selected = option_menu(
        menu_title=None,
        options=["Main Checker", "Dataset Insights"],
        icons=["shield-check", "database"],
        default_index=0,
        styles={
            "container": {"background-color": "#111827"},
            "nav-link": {"color": "white", "text-align": "left"},
            "nav-link-selected": {"background-color": "#374151"},
        }
    )

# ------------------------------------------------
# 4️⃣ Main Checker Page
# ------------------------------------------------
if selected == "Main Checker":
    st.title("🚫 Cyberbullying & Hate Speech Detection with Legal Awareness")

    # --- SEGMENT 1: Input & Suggestions ---
    st.markdown('<div class="main-card">', unsafe_allow_html=True)
    st.subheader(" Input Content")

    suggestions = {
        "Neutral": "The weather is quite nice today for a walk.",
        "Harsh": "You are a pathetic loser, I hate everything about you!",
        "Threatening": "I am going to find where you live and hurt you."
    }

    col_s1, col_s2, col_s3 = st.columns(3)
    if col_s1.button("Example: Neutral"): st.session_state.user_text = suggestions["Neutral"]
    if col_s2.button("Example: Harsh"): st.session_state.user_text = suggestions["Harsh"]
    if col_s3.button("Example: Threatening"): st.session_state.user_text = suggestions["Threatening"]

    user_input = st.text_area(
        "Enter text to analyze:",
        value=st.session_state.get("user_text", ""),
        placeholder="Type or click a suggestion above...",
        height=120
    )
    analyze_btn = st.button(" Run Analysis", use_container_width=True)
    st.markdown('</div>', unsafe_allow_html=True)

    # --- SEGMENT 2: Analysis Results ---
    if analyze_btn and user_input:
        st.markdown('<div class="main-card">', unsafe_allow_html=True)
        st.subheader("AI Analysis ")

        probs = predict(user_input)
        is_toxic = any(p > 0.5 for p in probs)

        if is_toxic:
            st.error(" **High Toxicity Detected:** This content may violate safety guidelines.")
        else:
            st.success("**Safe Content:** No significant toxicity detected.")

        fig = go.Figure()
        for label, prob in zip(labels, probs):
            fig.add_trace(go.Bar(
                y=[label], x=[prob],
                orientation='h',
                marker=dict(color="#b91c1c" if prob > 0.5 else "#10b981"),
                text=f"{prob:.2%}", textposition='outside'
            ))
        fig.update_layout(xaxis=dict(range=[0, 1.1]), height=350, margin=dict(l=0, r=0, t=0, b=0), showlegend=False)
        st.plotly_chart(fig, use_container_width=True)
        st.markdown('</div>', unsafe_allow_html=True)

    # --- SEGMENT 3: IPC Rules & Laws ---
    st.markdown('<div class="main-card">', unsafe_allow_html=True)
    st.subheader(" Indian Legal Framework & Punishments ⚖️")

    laws = [
        ("Section 66E (IT Act)", "Violation of privacy (publishing private images without consent).", "Up to 3 years imprisonment or ₹2 lakh fine."),
        ("Section 67 (IT Act)", "Transmitting obscene material in electronic form.", "Up to 3 years imprisonment + fine."),
        ("Section 507 (IPC)", "Criminal intimidation by anonymous communication.", "Up to 2 years imprisonment."),
        ("Section 509 (IPC)", "Word, gesture, or act intended to insult the modesty of a woman.", "Up to 3 years imprisonment + fine."),
        ("Section 499/500 (IPC)", "Defamation (harming reputation via digital comments).", "Up to 2 years imprisonment or fine.")
    ]

    for title, desc, punishment in laws:
        st.markdown(f"""
            <div class="legal-card">
                <h4 style='margin:0;'>{title}</h4>
                <p style='margin:5px 0;'><b>Offense:</b> {desc}</p>
                <p style='margin:0; color:#b91c1c;'><b>Punishment:</b> {punishment}</p>
            </div>
        """, unsafe_allow_html=True)
    st.markdown('</div>', unsafe_allow_html=True)

# ------------------------------------------------
# 5️⃣ Dataset Insights Page
# ------------------------------------------------
elif selected == "Dataset Insights":
    st.title("📈 Dataset Insights")
    st.write("the model is trained on the Jigsaw Toxic Comment Classification Challenge dataset.")

    # Mock data for visualization
    data = {"Category": labels, "Count": [15294, 1595, 8449, 478, 7877, 1405]}
    df = pd.DataFrame(data)

    fig_dataset = go.Figure([go.Pie(labels=df['Category'], values=df['Count'], hole=.3)])
    fig_dataset.update_layout(title="Training Set Label Distribution")
    st.plotly_chart(fig_dataset, use_container_width=True)

    st.info("This model uses BERT (Bidirectional Encoder Representations from Transformers) to understand the context of words in a sentence.")

2026-03-07 11:36:03.197 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-07 11:36:03.200 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-07 11:36:03.201 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-07 11:36:03.202 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-07 11:36:03.210 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-07 11:36:03.211 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-07 11:36:03.213 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-07 11:36:03.215 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [5]:
!pip install streamlit_option_menu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 21.5 MB/s eta 0:00:00


In [2]:
!pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 78.9 MB/s eta 0:00:00
